# SAADE v5 · 04 — Одноразовий фінальний hidden test

**Не запускайте до завершення matched REAL–TTS експерименту, перевірки незалежних голосів та фіксації моделей.** Відкриття тестової вибірки є незворотною подією дослідницького протоколу. Цей ноутбук не навчає моделі та не змінює пороги. Пілотний режим не допускається.

Після створення маркера `V5_HIDDEN_TEST_STARTED.json` друге виконання заблоковане. Не видаляйте маркер, навіть якщо запуск перервався.

## 0. Підключення

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys, json, shutil, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT / 'AUA'
RESULTS = PROJECT / 'results'
OUT = RESULTS / 'SAADE_AUA_2026_v5_FINAL'
assert AUA.is_dir(), f'Не знайдено AUA: {AUA}'
assert shutil.which('ffmpeg'), 'Потрібно встановити ffmpeg, щоб відтворити MP3-сценарії'
sys.path.insert(0,str(Path.cwd().resolve()))
import saade_final_core as S
import saade_final_data as D
import saade_final_methods as M
import saade_final_visuals as V
import saade_final_adaptive as A
for mod in (S,D,M,V,A):
    assert Path(mod.__file__).resolve().parent == Path.cwd().resolve(), f'Чужа версія модуля: {mod.__file__}'
cfg=S.Config(root=str(AUA),out=str(OUT),scenario_profile='v2_5level',
             seed=42,seconds=4.,min_seconds=2.,augmentation_probability=.50,
             batch_size=12,lr=.0005,epochs=15,use_cache=True)
print('AUA:',AUA,'\nRESULTS:',OUT)
print('Дані лише для читання. Melnyk_dus не використовується.')

## 1. Лише читання зафіксованої фінальної вибірки та перевірка готовності

In [ ]:
SPLITS=OUT/'V5_02_LOCKED_EXPERIMENT_SEGMENTS.csv'
assert SPLITS.exists()
splits=pd.read_csv(SPLITS,low_memory=False)
models=('spectrogram_cnn','raw_wave_cnn','mfcc_svm')
seeds=(42,123,777)
epochs=15
info,status=A.research_readiness(splits,cfg,mode='matched_tts',trained_seeds=seeds)
display(info);display(pd.DataFrame([status]).T)
print('Готовність:',status['candidate_for_strict_review'])

## 2. Зафіксувати ваги, пороги та протокол (тільки після перевірки)

In [ ]:
FREEZE_AFTER_REVIEW=False
if FREEZE_AFTER_REVIEW:
    assert status['candidate_for_strict_review'],'Дані не готові до фінального тесту'
    path=A.freeze_for_hidden_once(splits,cfg,models=models,seeds=seeds,
                                  epochs=epochs,mode='matched_tts',min_voice_groups=3)
    print('Протокол зафіксовано:',path)
else:
    print('Протокол поки не зафіксовано. Встановіть FREEZE_AFTER_REVIEW=True тільки після рецензування.')

## 3. Фінальне одноразове оцінювання (явна згода)

In [ ]:
RUN_HIDDEN_ONE_TIME=False
if RUN_HIDDEN_ONE_TIME:
    summary=A.run_hidden_once(splits,cfg,authorization=True)
    display(summary)
else:
    print('Hidden test закритий. RUN_HIDDEN_ONE_TIME=False.')

## Важливо

Результати hidden test не можна використовувати для змін E3, архітектур, порогів, групувань або сегментації з подальшим повторним тестуванням на тій самій вибірці.